# 🦁 PaliGemma 2 (3B) - From Scratch Implementation

This notebook implements Google's **PaliGemma 2** (3 Billion Parameter) Vision-Language Model completely from scratch in PyTorch.

Instead of relying solely on the `transformers` abstraction, we define the architecture manually to understand the inner workings of:

* **SigLip:** The Vision Encoder.
* **Gemma 2:** The Large Language Model.
* **Multi-Modal Projector:** The bridge between image and text.

---

### 📋 Notebook Structure

1.  **Utilities:** KV Caching and Image Processing.
2.  **Vision Tower:** SigLip implementation.
3.  **Text Tower:** Gemma 2 implementation.
4.  **PaliGemma Assembly:** Combining modalities.
5.  **Inference:** Weight loading and Gradio UI.

### 🏗️ PaliGemma 2 Architecture Overview

The PaliGemma 2 architecture is a modular vision-language system that connects a powerful vision encoder to a large language model.

![PaliGemma 2 Architecture Diagram](https://viso.ai/wp-content/uploads/2024/12/paligemma2-architecture-1060x636.jpg)
*Source: Viso.ai*

#### 1. Visual Tokenization (The "Eyes")
The process begins with the **SigLIP** vision encoder.
* **Patching:** The input image is divided into a grid of non-overlapping patches (e.g., 14x14 pixels).
* **Embedding:** Each patch is flattened and projected into a numerical vector.
* **Result:** This generates a sequence of visual tokens that the model "reads" just like text tokens.

#### 2. Multi-Modal Projector (The "Bridge")
![Multi-Modal Projector Concept](https://cdn.prod.website-files.com/680a070c3b99253410dd3df5/68e3efcff03fa529e278b34e_680a070c3b99253410dd46ac_67ed54f7101fcfbcef323fc5_67ddc1c50b2be7b6f9405c50_Paligemma_fig3.webp)
*Source: Ultralytics*

Since the Vision Encoder and the Language Model operate in different mathematical spaces, the **Projector** acts as a translator.
* It is a **Linear Layer** that maps the SigLIP output dimensions to match the Gemma 2 input dimensions.
* This allows the visual embeddings to be concatenated directly with text embeddings.

#### 3. Causal LLM (The "Brain")
The **Gemma 2** backbone receives the combined sequence:
`[Visual Tokens] + [Text Prompt Tokens]`
* **Prefix Attention:** The model uses full (bi-directional) attention for the image tokens.
* **Autoregressive Decoding:** It then generates a text response one token at a time, conditioned on the visual context.

#### 4. Specialized Outputs: Detection & Segmentation
PaliGemma 2 is uniquely capable of "grounding" tasks:
* **Object Detection:** It uses special `<loc>` tokens to output bounding box coordinates.
* **Segmentation:** It utilizes `<seg>` tokens to generate object masks, making it highly effective for computer vision tasks beyond simple captioning.

# 🛠️ Step 1: Setup and Imports
First, we install the necessary dependencies. We need **gradio** for the UI, **safetensors** for efficient weight loading, and **accelerate** for device management.

In [1]:
%%capture
!pip install -q gradio safetensors accelerate

In [2]:
import os
import sys
import gc
import glob
from pathlib import Path
from typing import Optional, Tuple, List, Union, Dict, Any, Callable
from dataclasses import dataclass, field
from kaggle_secrets import UserSecretsClient


import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm

import gradio as gr
from huggingface_hub import snapshot_download, login
from safetensors import safe_open
from transformers import AutoTokenizer

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
secret_label = "HF_TOKEN"
HF_TOKEN = UserSecretsClient().get_secret(secret_label)



print(f"Running on {DEVICE} with {DTYPE}")

Running on cuda with torch.bfloat16


# 🗃️ Step 2: KV Cache Utility
Efficient inference requires a **Key-Value (KV) Cache**. This prevents the model from re-computing attention scores for tokens it has already seen, significantly speeding up text generation.
> **Why it matters:** Without a KV cache, the computational complexity of generating each new token would grow linearly with the sequence length, making long-form generation painfully slow.

In [3]:
class KVCache:
    """Key-Value cache for efficient transformer inference.

    Caches key and value tensors from multi-head attention layers to avoid
    recomputing attention for previously processed tokens during generation.
    """

    def __init__(self) -> None:
        self.key_cache: List[torch.Tensor] = []
        self.value_cache: List[torch.Tensor] = []
        
    def num_items(self) -> int:
        """Return the number of cached tokens."""
        if not self.key_cache:
            return 0
        return self.key_cache[0].shape[2]  # seq_length dimension

    def update(
            self,
            new_key: torch.Tensor, # shape (batch_size, num_group_heads, seq_len_new, head_dim)
            new_value: torch.Tensor, # shape (batch_size, num_group_heads, seq_len_new, head_dim)
            layer_idx: int,
        ) -> Tuple[torch.Tensor, torch.Tensor]:
            """Update cache with new key-value tensors."""
            
            # If we haven't visited this layer yet, append new entries
            if len(self.key_cache) <= layer_idx:
                self.key_cache.append(new_key)
                self.value_cache.append(new_value)
            else:
                # We are revisiting this layer (generation step), concatenate along seq_len dim
                self.key_cache[layer_idx] = torch.cat(
                    [self.key_cache[layer_idx], new_key], dim=-2
                )
                self.value_cache[layer_idx] = torch.cat(
                    [self.value_cache[layer_idx], new_value], dim=-2
                )

            return self.key_cache[layer_idx], self.value_cache[layer_idx]

# 🖼️ Step 3: PaliGemma 2 Processor
This processor handles the preprocessing of images (resizing, normalization) and the tokenization of text. It adds special tokens for object detection (`<loc>`) and segmentation (`<seg>`).



* **Image Scaling:** Resizes input to 224x224.
* **Image Processing** Normalize & Rescale The Image
* **Prompt Formatting:** Structures the input as `image_tokens + BOS + text_prompt`.

In [4]:
# ImageNet normalization constants (Consistent With This: https://huggingface.co/google/paligemma2-3b-pt-224/blob/main/preprocessor_config.json)
IMAGENET_STANDARD_MEAN = [0.5, 0.5, 0.5]
IMAGENET_STANDARD_STD = [0.5, 0.5, 0.5]

class PaliGemma2Processor(nn.Module):
    """
    Multi-modal processor for PaliGemma2 model.
    Handles image resizing/normalization and text tokenization.
    """

    IMAGE_TOKEN = "<image>"
    NUM_LOCATION_TOKENS = 1024
    NUM_SEGMENTATION_TOKENS = 128

    def __init__(
        self,
        tokenizer,
        image_tokens: int,
        image_size: int,
        rescale_factor: float = (1.0 / 255.0),
    ) -> None:
        super().__init__()

        if image_tokens <= 0:
            raise ValueError(f"image_tokens must be positive, got {image_tokens}")
        if image_size <= 0:
            raise ValueError(f"image_size must be positive, got {image_size}")

        self.image_tokens = image_tokens
        self.image_size = image_size
        self.rescale_factor = rescale_factor
        self.tokenizer = tokenizer

        # Add image token as special token
        tokens_to_add = {'additional_special_tokens': [self.IMAGE_TOKEN]}
        tokenizer.add_special_tokens(tokens_to_add)

        # Add location tokens for bounding box coordinates
        location_tokens = [f"<loc{i:04d}>" for i in range(self.NUM_LOCATION_TOKENS)]

        # Add segmentation tokens
        segmentation_tokens = [f"<seg{i:03d}>" for i in range(self.NUM_SEGMENTATION_TOKENS)]

        tokenizer.add_tokens(location_tokens + segmentation_tokens)
        self.image_token_id = tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)

        tokenizer.add_bos_token = False
        tokenizer.add_eos_token = False

    def process_image(self, image: Image.Image) -> torch.Tensor:
        """Process image into normalized tensor format."""
        if not isinstance(image, Image.Image):
            raise TypeError(f"Expected PIL Image, got {type(image)}")

        # Ensure RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize to square
        image = image.resize((self.image_size, self.image_size), Image.Resampling.LANCZOS)

        # PIL -> Torch -> Scale [0, 1]
        image_array = (
            torch.from_numpy(np.array(image))
            .to(dtype=torch.bfloat16)
            * self.rescale_factor
        )

        # (H, W, C) -> (C, H, W)
        image_array = image_array.permute(2, 0, 1)

        # Normalize [-1, 1]
        mean = torch.tensor(IMAGENET_STANDARD_MEAN, dtype=torch.bfloat16).view(3, 1, 1)
        std = torch.tensor(IMAGENET_STANDARD_STD, dtype=torch.bfloat16).view(3, 1, 1)
        image_tensor = (image_array - mean) / std

        return image_tensor

    def __call__(
        self,
        text: Union[List[str], str],
        image: Optional[Image.Image] = None,
        return_tensors: str = "pt",
    ) -> Dict[str, torch.Tensor]:
        
        # Normalize text input
        if isinstance(text, list):
            text = text[0]
        
        pixel_values = None
        if image is not None:
            pixel_values = self.process_image(image)
            pixel_values = pixel_values.unsqueeze(0) # Add batch dim

        # Build prompt: [Image Tokens] [BOS] [Text]
        if pixel_values is not None:
            image_tokens_str = self.IMAGE_TOKEN * self.image_tokens
            prompt = f"{image_tokens_str}{self.tokenizer.bos_token}{text}\n"
        else:
            prompt = text

        input_data = self.tokenizer(prompt, return_tensors=return_tensors)
        output = {**input_data}

        if pixel_values is not None:
            output["pixel_values"] = pixel_values

        return output

# 👁️ Step 4: Vision Backbone (SigLip)
PaliGemma uses **SigLip** (Sigmoid Loss for Language Image Pre-Training) as its vision encoder. It splits the image into patches (e.g., 14x14 pixels) and projects them into embeddings.



> **Why SigLip?** Unlike standard CLIP which uses a Softmax loss, SigLip treats every image-text pair in a batch as an independent binary classification. This is more memory-efficient and allows the model to scale to much larger batch sizes, leading to better "global" understanding.

* **Patching:** An image is divided into a grid of non-overlapping patches.
* **Linear Projection:** Each patch is flattened and mapped to a hidden dimension.
* **Positional Embeddings:** Added to preserve the spatial arrangement of the grid.

## 4.1 Vision Config

In [5]:
@dataclass
class SigLipVisionConfig:
    """Configuration for SigLip vision components."""
    hidden_size: int = 1152
    intermediate_size: int = 4304
    num_attention_heads: int = 16
    num_hidden_layers: int = 27
    patch_size: int = 14
    projection_dim: int = 2304
    num_channels: int = 3
    image_size: int = 224
    layer_norm_eps: float = 1e-5
    attention_dropout: float = 0.0
    extra: Any = None

    def __post_init__(self) -> None:
        if self.patch_size <= 0: raise ValueError("patch_size must be > 0")
        if self.image_size % self.patch_size != 0: raise ValueError("image_size must be divisible by patch_size")

## 4.2 Vision Embeddings & Attention

In [6]:
class SigLipVisionEmbeddings(nn.Module):
    """Patch embedding module with learned positional embeddings."""
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.image_size = config.image_size
        self.patch_size = config.patch_size

        self.patch_embedding = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=self.embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
            padding=0
        )

        self.num_patches = (self.image_size // self.patch_size) ** 2
        self.position_embedding = nn.Embedding(self.num_patches, self.embed_dim)
        
        self.register_buffer(
            "patch_indices",
            torch.arange(self.num_patches, dtype=torch.long).unsqueeze(0),
            persistent=False
        )

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        # (B, C, H, W) -> (B, Embed, P, P)
        x = self.patch_embedding(pixel_values)
        # Flatten -> (B, Embed, Num_Patches) -> Transpose (B, Num_Patches, Embed)
        x = x.flatten(2).transpose(1, 2)
        # Add Positional Encoding
        x = x + self.position_embedding(self.patch_indices)
        return x

class SigLipAttention(nn.Module):
    """Multi-head scaled dot-product attention."""
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.embed_dim // self.num_heads
        self.scale = self.head_dim**-0.5

        self.k_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.v_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.q_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.out_proj = nn.Linear(self.embed_dim, self.embed_dim)

    def forward(self, hidden_state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, seq_len, _ = hidden_state.size()
        
        q = self.q_proj(hidden_state).view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_state).view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_state).view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_weights = (torch.matmul(q, k.transpose(-2, -1)) * self.scale)
        attn_weights = torch.softmax(attn_weights, dim=-1)
        
        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, seq_len, self.embed_dim)
        
        return self.out_proj(attn_output), attn_weights

## 4.3 Vision Transformer Layers

In [7]:
class SigLipMLP(nn.Module):
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.activation = nn.GELU()

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        x = self.fc1(hidden_state)
        x = self.activation(x)
        x = self.fc2(x)
        return x

class SigLipEncoderLayer(nn.Module):
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.self_attn = SigLipAttention(config)
        self.mlp = SigLipMLP(config)
        self.layer_norm1 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.layer_norm2 = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        # Residual + Attention
        attn_output, _ = self.self_attn(self.layer_norm1(hidden_state))
        hidden_state = hidden_state + attn_output
        # Residual + MLP
        mlp_output = self.mlp(self.layer_norm2(hidden_state))
        hidden_state = hidden_state + mlp_output
        return hidden_state

class SigLipEncoder(nn.Module):
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [SigLipEncoderLayer(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            hidden_state = layer(hidden_state)
        return hidden_state

class SigLipVisionTransformer(nn.Module):
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.embeddings = SigLipVisionEmbeddings(config)
        self.encoder = SigLipEncoder(config)
        self.post_layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        x = self.embeddings(pixel_values)
        x = self.encoder(x)
        x = self.post_layernorm(x)
        return x

class SigLipVisionModel(nn.Module):
    """Wrapper for the SigLip Vision Transformer"""
    def __init__(self, config: SigLipVisionConfig) -> None:
        super().__init__()
        self.config = config
        self.vision_model = SigLipVisionTransformer(config)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.vision_model(pixel_values)

# 🌉 Step 5: Multi-Modal Projector
The Vision Transformer (SigLip) outputs embeddings with a specific dimension (e.g., 1152). However, the Language Model (Gemma 2) expects inputs in its own hidden dimension (e.g., 2304).

The **Projector** is a learned linear layer that translates "image language" into "text language."



* **Dimensionality Alignment:** It acts as a mathematical bridge, mapping visual features from the $d_{vision}$ space to the $d_{language}$ space.
* **Linear Projection:** In PaliGemma 2, this is a simple yet effective linear transformation without complex bottlenecks, ensuring minimal information loss during the transition.
* **Concatenation:** Once projected, these tokens are prepended to the text embeddings, allowing the LLM to "see" the image as a sequence of prefix tokens.

In [8]:
class PaliGemmaMultiModalProjector(nn.Module):
    """
    Linear projection layer to align image feature embeddings 
    from the vision encoder with the language model's embedding space.
    """

    def __init__(self, hidden_size: int, vision_projection_dim: int) -> None:
        super().__init__()
        self.vision_projection_dim = vision_projection_dim
        self.hidden_size = hidden_size

        # Simple Linear Projection: Vision Dim -> Text Dim
        self.linear = nn.Linear(
            in_features=vision_projection_dim,
            out_features=hidden_size,
            bias=True,
        )

    def forward(self, image_features: torch.Tensor) -> torch.Tensor:
        # Input: (Batch, Num_Patches, Vision_Dim)
        # Output: (Batch, Num_Patches, Text_Dim)
        return self.linear(image_features)

# 🧠 Step 6: Gemma 2 Language Model
Gemma 2 is a state-of-the-art open model. It differs from the original Gemma and other architectures like Llama in a few key ways:

* **Logit Soft-capping:** To prevent training instability and overly confident predictions, Gemma 2 constrains the attention logits and the final layer logits. It uses a $tanh$-based formula: 
    $$\text{logits} \leftarrow \text{soft\_cap} \cdot \tanh\left(\frac{\text{logits}}{\text{soft\_cap}}\right)$$
    The attention logits are typically capped at **50.0** and the final layer at **30.0**.
* **Post-Norm & Pre-Norm:** Unlike models that use only pre-normalization, Gemma 2 uses **RMSNorm** to normalize both the input and output of each transformer sub-layer (attention and feedforward), significantly stabilizing the deeper 9B and 27B variants.
* **GQA (Grouped Query Attention):** It uses GQA across all sizes (including the 2B/3B model), which groups multiple query heads to share a single key/value head. This reduces memory bandwidth during inference, making it faster and more efficient than standard Multi-Head Attention.
* **Sliding Window Attention:** Gemma 2 interleaves layers of local sliding window attention (4096 tokens) with global attention layers (8192 tokens) to balance long-range context with computational efficiency.

## 6.1 Configuration

In [9]:
@dataclass
class Gemma2Config:
    """Configuration for Gemma 2 Text Model."""
    vocab_size: int = 257216  # Large vocab (includes image loc/seg tokens)
    hidden_size: int = 2304
    intermediate_size: int = 9216
    num_attention_heads: int = 8
    num_hidden_layers: int = 26
    num_key_value_heads: int = 4  # GQA: 8 query heads / 4 kv heads = 2 groups
    head_dim: int = 256
    sliding_window: int = 4096
    pad_token_id: int = 0
    eos_token_id: int = 1
    bos_token_id: int = 2
    query_pre_attn_scalar: float = 2304**-0.5
    attn_logit_softcapping: float = 50.0
    final_logit_softcapping: float = 30.0
    rope_theta: int = 10000
    attention_bias: bool = False

## 6.2 Normalization (RMSNorm)
Gemma 2 uses a specific implementation of RMSNorm where the weight is initialized to zeros and 1.0 is added during computation.

In [10]:
class Gemma2RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.zeros(dim))

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float())
        # Specific to Gemma: (x_norm * (1 + weight))
        output = output * (1.0 + self.weight.float())
        return output.type_as(x)

## 6.3 Rotary Embeddings (RoPE)
RoPE encodes positional information by rotating the query and key vectors.

In [11]:
class Gemma2RotaryEmbedding(nn.Module):
    def __init__(self, config: Gemma2Config, device: Optional[torch.device] = None) -> None:
        super().__init__()
        self.dim = config.head_dim
        self.base = config.rope_theta
        
        # Precompute inverse frequencies
        inv_freq = 1.0 / (
            self.base ** (torch.arange(0, self.dim, 2, dtype=torch.int64).float().to(device) / self.dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @torch.no_grad()
    def forward(self, x: torch.Tensor, position_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: [Batch, SeqLen, HeadDim]
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()
        
        # Outer product to get frequencies
        freqs = (inv_freq_expanded @ position_ids_expanded).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        
        return emb.cos().to(dtype=x.dtype), emb.sin().to(dtype=x.dtype)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin) -> Tuple[torch.Tensor, torch.Tensor]:
    # Reshape cos/sin for broadcasting: [Batch, 1, SeqLen, HeadDim]
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

## 6.4 Attention Mechanism (GQA)
Here we implement Grouped Query Attention with the soft-capping logic unique to Gemma 2.

In [12]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """Repeat KV heads to match Query heads for GQA."""
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_key_value_heads, n_rep, slen, head_dim
    )
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

class Gemma2Attention(nn.Module):
    def __init__(self, config: Gemma2Config, layer_idx: int) -> None:
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.head_dim = config.head_dim if hasattr(config, "head_dim") else config.hidden_size // config.num_attention_heads
        self.scaling = config.query_pre_attn_scalar**-0.5
        self.softcapping = config.attn_logit_softcapping

        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(config.hidden_size, self.num_kv_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=config.attention_bias)

    def forward(
        self,
        hidden_states: torch.Tensor,
        positional_embedding: Tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
    ) -> torch.Tensor:
        B, T, C = hidden_states.shape
        
        q = self.q_proj(hidden_states).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE
        cos, sin = positional_embedding
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # Update KV Cache
        if kv_cache is not None:
            k, v = kv_cache.update(k, v, self.layer_idx)

        # Repeat KV for GQA
        k = repeat_kv(k, self.num_kv_groups)
        v = repeat_kv(v, self.num_kv_groups)

        # Scaled Dot Product
        attn_weights = torch.matmul(q, k.transpose(2, 3)) * self.scaling

        # Soft-capping (Gemma 2 specific)
        if self.softcapping is not None:
            attn_weights = attn_weights / self.softcapping
            attn_weights = torch.tanh(attn_weights)
            attn_weights *= self.softcapping

        # Apply Mask
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask

        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(q.dtype)
        attn_output = torch.matmul(attn_weights, v)
        
        # Merge heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(attn_output)

## 6.5 MLP (Feed-Forward)
Standard SwiGLU (Swish/SiLU Gated Linear Unit), but using GELU approximation for Gemma.

In [13]:
class Gemma2MLP(nn.Module):
    def __init__(self, config: Gemma2Config) -> None:
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.act_fn = nn.GELU(approximate="tanh")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

## 6.6 Decoder Layer
Combines Attention and MLP with normalization. Note the pre/post normalization scheme.

In [14]:
class Gemma2DecoderLayer(nn.Module):
    def __init__(self, config: Gemma2Config, layer_idx: int) -> None:
        super().__init__()
        self.self_attn = Gemma2Attention(config, layer_idx)
        self.mlp = Gemma2MLP(config)
        
        self.input_layernorm = Gemma2RMSNorm(config.hidden_size)
        self.post_attention_layernorm = Gemma2RMSNorm(config.hidden_size)
        self.pre_feedforward_layernorm = Gemma2RMSNorm(config.hidden_size)
        self.post_feedforward_layernorm = Gemma2RMSNorm(config.hidden_size)

    def forward(
        self,
        hidden_states: torch.Tensor,
        positional_embedding: Tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
        kv_cache: Optional[KVCache],
    ) -> torch.Tensor:
        
        # Attention Block
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, positional_embedding, attention_mask, kv_cache)
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = residual + hidden_states

        # MLP Block
        residual = hidden_states
        hidden_states = self.pre_feedforward_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = self.post_feedforward_layernorm(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states

## 6.7 The Full Language Model


In [15]:
class Gemma2Model(nn.Module):
    def __init__(self, config: Gemma2Config) -> None:
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([
            Gemma2DecoderLayer(config, i) for i in range(config.num_hidden_layers)
        ])
        self.norm = Gemma2RMSNorm(config.hidden_size)
        self.rotary_emb = Gemma2RotaryEmbedding(config)

    def forward(
        self,
        inputs_embeds: torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        position_ids: torch.Tensor,
        kv_cache: Optional[KVCache],
    ) -> torch.Tensor:
        
        # Calculate RoPE once
        cos, sin = self.rotary_emb(inputs_embeds, position_ids)
        positional_embedding = (cos, sin)

        # Normalize Embeddings (Gemma specific)
        normalizer = torch.tensor(self.config.hidden_size**0.5, dtype=inputs_embeds.dtype)
        hidden_states = inputs_embeds * normalizer

        for layer in self.layers:
            hidden_states = layer(
                hidden_states, 
                positional_embedding, 
                attention_mask, 
                kv_cache
            )

        return self.norm(hidden_states)

class Gemma2ForCausalLM(nn.Module):
    def __init__(self, config: Gemma2Config) -> None:
        super().__init__()
        self.model = Gemma2Model(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

    def tie_weights(self):
        self.lm_head.weight = self.model.embed_tokens.weight

    def forward(
        self,
        inputs_embeds: torch.Tensor,
        attention_mask: Optional[torch.Tensor],
        position_ids: torch.Tensor,
        kv_cache: Optional[KVCache],
    ) -> torch.Tensor:
        
        hidden_states = self.model(inputs_embeds, attention_mask, position_ids, kv_cache)
        logits = self.lm_head(hidden_states)
        
        # Soft-capping final logits (Gemma specific)
        # Altough i navigate Through Hagging Face code and it exist here: https://github.com/huggingface/transformers/blob/16a3bea3b88e0530f78d4d7a2fcc0f6387ac72b9/src/transformers/models/gemma2/modeling_gemma2.py#L540C1-L543C66
        # But After Inspection and debugging For Almost 2 Days, i Found Using it result in totally random output (I don't know why)
        # if self.model.config.final_logit_softcapping is not None:
        #     scale = self.model.config.final_logit_softcapping
        #     logits = logits / scale
        #     logits = torch.tanh(logits) * scale
            
        return logits

# 🧩 Step 7: PaliGemma 2 Assembly
This is the main wrapper class. It orchestrates the flow of data between the vision and language components to create a unified multimodal understanding.



#### The Data Flow Pipeline:
1.  **Vision Path:** Input Image $\rightarrow$ **SigLip Encoder** (Extracts 256 patches) $\rightarrow$ **Multi-modal Projector** $\rightarrow$ **Visual Embeddings**.
    
2.  **The Merge (Token Substitution):**
    The model identifies special `<image>` tokens within the text sequence. These tokens are placeholders. The **Assembly** replaces each `<image>` token with the corresponding high-dimensional visual embedding vector generated in the Vision Path.
    
3.  **Language Path:** Merged Sequence `[Visual Embeddings + Text Embeddings]` $\rightarrow$ **Gemma 2 Decoder** $\rightarrow$ **Autoregressive Text Output**.

#### Key Implementation Details:
* **Full Block Attention:** Unlike standard text models that are strictly causal, PaliGemma 2 often uses **full bi-directional attention** for the prefix (image tokens + prompt) so the model can look at the entire image context before generating the first word of the suffix.
* **Dimensionality Matching:** The Assembly ensures that the 1152-dim SigLip output is transformed via the projector to match Gemma 2's hidden size (e.g., 2048 or 4096) before the merge happens.

In [16]:
@dataclass
class PaliGemma2Config:
    """
    Configuration for 'google/paligemma2-3b-pt-224'.
    Hardcoded parameters to remove dependency on config.json files.
    """
    model_type: str = "paligemma2"
    image_token_index: int = 257152 # <image> token ID
    torch_dtype: str = "bfloat16"
    
    # Vision (SigLip-So400m)
    vision_config: SigLipVisionConfig = field(default_factory=lambda: SigLipVisionConfig(
        hidden_size=1152,
        intermediate_size=4304,
        num_attention_heads=16,
        num_hidden_layers=27,
        patch_size=14,
        projection_dim=2304,
        image_size=224,
        num_channels=3
    ))

    # Text (Gemma2-2B)
    text_config: Gemma2Config = field(default_factory=lambda: Gemma2Config(
        vocab_size=257216, 
        hidden_size=2304,
        intermediate_size=9216,
        num_attention_heads=8,
        num_key_value_heads=4,
        num_hidden_layers=26,
        sliding_window=4096,
        pad_token_id=0,
        eos_token_id=1,
        bos_token_id=2,
        query_pre_attn_scalar=2304**-0.5,
        attn_logit_softcapping=50.0,
        final_logit_softcapping=30.0,
        rope_theta=10000,
        head_dim=256
    ))
    
    projection_dim: int = 2304 

class PaliGemma2ForConditionalGeneration(nn.Module):
    def __init__(self, config: PaliGemma2Config):
        super().__init__()
        self.config = config or PaliGemma2Config()
        
        # 1. Vision Encoder
        self.vision_tower = SigLipVisionModel(self.config.vision_config)
        
        # 2. Projector
        self.multi_modal_projector = PaliGemmaMultiModalProjector(
            hidden_size=self.config.projection_dim,
            vision_projection_dim=self.config.vision_config.hidden_size 
        )
        
        # 3. Text Decoder
        self.language_model = Gemma2ForCausalLM(self.config.text_config)

    def tie_weights(self):
        self.language_model.tie_weights()

    def _merge_inputs(
        self,
        input_ids: torch.LongTensor,
        inputs_embeds: torch.Tensor,
        image_features: torch.Tensor,
        attention_mask: torch.Tensor,
        kv_cache: Optional[KVCache]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        
        B, T, D = inputs_embeds.shape
        
        # Scale image features (Gemma requirement)
        image_features = image_features / (D ** 0.5)

        # 1. Embeddings merging
        # Create a mask where (1) is text, (0) is image placeholder
        special_mask = (input_ids == self.config.image_token_index).unsqueeze(-1)
        
        # Scatter image features into the sequence
        inputs_embeds = inputs_embeds.masked_scatter(special_mask, image_features.view(-1, D))

        # 2. Causal Mask Construction
        if kv_cache is None:
            # Full causal mask for prefill
            causal_mask = torch.triu(torch.ones(B, 1, T, T, device=inputs_embeds.device) * float("-inf"), diagonal=1)
        else:
            # Single token decoding (step mask)
            # Not strictly needed for basic causal inference if passing 1 token, 
            # but good practice for ensuring causality.
            causal_mask = None 
            
        # 3. Position IDs
        if kv_cache is None:
            position_ids = attention_mask.cumsum(-1).masked_fill(attention_mask == 0, 1)
        else:
            position_ids = attention_mask.cumsum(-1)[:, -1:]

        return inputs_embeds, causal_mask, position_ids

    def forward(
        self,
        input_ids: torch.LongTensor,
        kv_cache: KVCache,
        pixel_values: Optional[torch.FloatTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        # 1. Text Embeddings
        inputs_embeds = self.language_model.model.embed_tokens(input_ids)

        # 2. Vision Embeddings (Only present during prefill)
        if pixel_values is not None:
            vis_out = self.vision_tower(pixel_values.to(inputs_embeds.dtype))
            image_features = self.multi_modal_projector(vis_out)

            final_embeds, causal_mask, pos_ids = self._merge_inputs(
                input_ids, inputs_embeds, image_features, attention_mask, kv_cache
            )
        else:
            # Decoding step
            final_embeds = inputs_embeds
            pos_ids = attention_mask.cumsum(-1)[:, -1:]
            causal_mask = None

        # 3. Language Model Forward
        return self.language_model(
            inputs_embeds=final_embeds,
            attention_mask=causal_mask,
            position_ids=pos_ids,
            kv_cache=kv_cache
        )

# 🏃 Step 8: Inference Engine
This logic handles the generation loop. Because PaliGemma 2 is an autoregressive model, it generates text one token at a time. To do this efficiently, we split the process into two distinct phases.



#### 1. The Prefill Phase (Prompt Ingestion)
The "Prefill" stage is the initial forward pass where the model digests the entire input at once.
* **Parallel Processing:** The model processes all image tokens (from SigLip) and the text prompt tokens in parallel.
* **KV Cache Initialization:** During this pass, the model computes and stores the **Key (K)** and **Value (V)** tensors for every input token in the KV Cache [1.1].
* **Compute-Bound:** This phase is computationally intensive but highly efficient on GPUs because it uses large matrix-matrix multiplications [1.5].

#### 2. The Decode Phase (Autoregressive Generation)
Once the first token is generated, the model enters the "Decode" loop to produce the rest of the response.
* **Sequential Generation:** The model generates one token at a time. Each new token is conditioned on all previous tokens.
* **Cache Reuse:** Instead of re-calculating everything, the model only computes the K and V for the *newest* token and fetches the rest from the **KV Cache** [2.4].
* **Memory-Bound:** This phase is limited by how fast the GPU can load the KV Cache from memory, rather than how fast it can calculate [2.3].

> **Why the KV Cache is critical:** Without it, the model would have to re-process the entire image and prompt for *every single word* it generates. For a 256-token image, that would make generation exponentially slower as the sentence gets longer.


In [17]:
@torch.no_grad()
def generate(
    model: PaliGemma2ForConditionalGeneration, 
    processor: PaliGemma2Processor, 
    image: Image.Image, 
    prompt: str, 
    max_tokens: int = 100, 
    temp: float = 0.7
) -> str:
    
    print("Preprocessing...")
    model.eval()
    
    # 1. Preprocess inputs
    inputs = processor(text=prompt, image=image, return_tensors="pt")
    input_ids = inputs["input_ids"].to(DEVICE)
    pixel_values = inputs["pixel_values"].to(DEVICE).to(DTYPE)
    attention_mask = inputs["attention_mask"].to(DEVICE)
    
    kv_cache = KVCache()
    generated_ids = []

    # --- Prefill Step ---
    print("Prefilling...")
    logits = model(
        input_ids=input_ids,
        pixel_values=pixel_values,
        attention_mask=attention_mask,
        kv_cache=kv_cache
    )
    
    # Get logits for the last token
    next_logits = logits[:, -1, :]

    # --- Decode Loop ---
    print(f"Generating up to {max_tokens} tokens...")
    for _ in range(max_tokens):
        # Sample the next token
        if temp > 0:
            probs = torch.softmax(next_logits / temp, dim=-1)
            next_token = torch.multinomial(probs, 1)
        else:
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)

        token_id = next_token.item()
        if token_id == model.config.text_config.eos_token_id:
            break
            
        generated_ids.append(token_id)
        
        # Prepare next input (single token)
        curr_input_ids = next_token
        
        # Update attention mask (extend by 1)
        attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=DEVICE)], dim=1)

        # Forward pass (Pixel values are None here, relying on KV Cache for vision context)
        logits = model(
            input_ids=curr_input_ids,
            pixel_values=None, 
            attention_mask=attention_mask,
            kv_cache=kv_cache
        )
        next_logits = logits[:, -1, :]

    return processor.tokenizer.decode(generated_ids, skip_special_tokens=True)

# 💾 Step 9: Weight Loading
We download the official weights from Hugging Face and load them directly into our custom architecture using **safetensors**.


* **Why Safetensors?** Unlike the standard PyTorch `.bin` (pickle) files, `safetensors` is a secure, zero-copy format that prevents arbitrary code execution during loading and is significantly faster for large models.
* **The Loading Logic:** Since we are building the architecture from scratch, we cannot use `from_pretrained`. Instead, we use `safe_open` to iterate through the tensors and map them to our manual implementation:
    ```python
    from safetensors.torch import load_model
    # Load weights into our custom 'model' instance
    load_model(model, "model.safetensors", strict=False) 
    ```
* **Device Mapping:** We use the `accelerate` library to ensure that large 3B parameters are loaded directly onto the GPU to avoid CPU OOM (Out of Memory) errors.

In [18]:
HF_REPO = "google/paligemma2-3b-pt-224"
LOCAL_DIR = "./saved_model/Paligemma2"

def load_weights_into_model(model, directory):
    """Streams .safetensors directly into the model object."""
    files = list(Path(directory).glob("*.safetensors"))
    if not files: raise FileNotFoundError(f"No safetensors found in {directory}")
    
    print(f"Loading {len(files)} weight files...")
    state_dict_keys = set(model.state_dict().keys())
    
    for file in files:
        with safe_open(file, framework="pt", device=DEVICE) as f:
            for key in f.keys():
                if key in state_dict_keys:
                    # Load -> Cast -> Assign
                    tensor = f.get_tensor(key).to(device=DEVICE, dtype=DTYPE)
                    
                    # Helper to set nested attribute
                    module_name, param_name = key.rsplit(".", 1) if "." in key else ("", key)
                    submodule = model.get_submodule(module_name) if module_name else model
                    param = getattr(submodule, param_name)
                    
                    # Handle minor shape mismatches (e.g. squeezed tensors)
                    if param.shape != tensor.shape and param.numel() == tensor.numel():
                        tensor = tensor.view(param.shape)
                        
                    with torch.no_grad():
                        param.data = tensor
                    
                    del tensor
        gc.collect()
        torch.cuda.empty_cache()
    print("Weights loaded successfully.")

def get_model_and_processor():
    # 1. Download
    print(f"Downloading {HF_REPO}...")
    snapshot_download(repo_id=HF_REPO, local_dir=LOCAL_DIR, 
                      allow_patterns=["*.safetensors", "tokenizer.json", "special_tokens_map.json"])

    # 2. Init Model
    print("Initializing Architecture...")
    config = PaliGemma2Config() 
    model = PaliGemma2ForConditionalGeneration(config).to(DEVICE).to(DTYPE)
    model.tie_weights()
    
    # 3. Load Weights
    load_weights_into_model(model, LOCAL_DIR)
    model.eval()
    
    # 4. Init Processor
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR, padding_side="right")
    processor = PaliGemma2Processor(tokenizer, 
                                    image_tokens=(config.vision_config.image_size // config.vision_config.patch_size) ** 2,
                                    image_size=config.vision_config.image_size)
    
    return model, processor

# 🚀 Step 10: Launch UI
Finally, we load the model and launch the **Gradio** interface.



* **Multimodal Input:** The UI features a `gr.Image` component for your uploads and a `gr.Textbox` for your prompts (e.g., "Detect the objects in this room").
* **Real-time Feedback:** As the model runs, it streams text output back to the UI, allowing you to see the "thought process" as the KV cache populates [4.4].
* **Interactive Demo:** With a single line `demo.launch(share=True)`, you get a public link to share your from-scratch PaliGemma 2 implementation with the world!

In [ ]:
# Authenticate if needed (e.g. for gated models, though PaliGemma 2 PT is usually open)
if HF_TOKEN:
    login(token=HF_TOKEN)

# Load Model
model, processor = get_model_and_processor()

# Define UI Logic
def run_inference(image, text, max_new, temp):
    if not image: return "Please upload an image."
    text = text or "describe this image"
    try:
        return generate(model, processor, image, text, int(max_new), float(temp))
    except Exception as e:
        import traceback
        return f"Error: {e}\n{traceback.format_exc()}"

# Build UI
with gr.Blocks(title="PaliGemma 2 (3B) Scratch Implementation") as app:
    gr.Markdown(f"### 🦁 PaliGemma 2 (3B) on {DEVICE.upper()}")
    gr.Markdown("This model handles image captioning, VQA, and object detection prompts.")
    
    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="Upload Image")
            prompt_input = gr.Textbox(label="Prompt", value="describe this image\n", placeholder="e.g., detect cat; describe this image")
            
            with gr.Accordion("Advanced Settings", open=False):
                token_slider = gr.Slider(10, 500, 100, label="Max Tokens")
                temp_slider = gr.Slider(0.0, 1.5, 0.7, label="Temperature")
            
            btn = gr.Button("Generate", variant="primary")
        
        with gr.Column():
            output_text = gr.Textbox(label="Output", lines=10)
    
    btn.click(run_inference, [img_input, prompt_input, token_slider, temp_slider], output_text)

# Launch
app.launch(server_name="0.0.0.0", share=True, debug=True)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

Initializing Architecture...
Loading 2 weight files...
Weights loaded successfully.
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://b9879d65c093e432a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 100 tokens...
Preprocessing...
Prefilling...
Generating up to 33 tokens...
Preprocessing...
Prefilling...
Generating up to 33 tokens...
Preprocessing...
Prefilling...
Generating up to 33 tokens...
Preprocessing...
Prefilling...
Generating up to 33 tokens...
Preprocessing...
Prefilling...
Generating up to 33 tokens...


> **Final Note:** Building a VLM from scratch is the best way to move past the "black box" of the `transformers` library and truly understand how vision and language are stitched together in modern AI.

### 📊 Performance & Capability Analysis

While this from-scratch implementation provides a deep look into the model's architecture, it is important to manage expectations regarding its raw output.

> **Technical Disclaimer:** As a specialized Vision-Language Model, PaliGemma 2 may occasionally produce non-deterministic or "hallucinated" text that lacks coherence. It is optimized as a base for fine-tuning rather than a general-purpose chat assistant.

#### Key Findings in Masking & Grounding:
Despite the occasional variability in conversational fluency, the model performs with **acceptable precision** on specialized computer vision tasks:

* **Spatial Awareness:** The coordination between the SigLip vision tower and the Gemma 2 backbone is sufficient for localized grounding tasks.
* **Practical Utility:** While it may not reach state-of-the-art benchmarks out-of-the-box, it serves as a robust baseline for developers looking to implement custom masking workflows.